In [1]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("./data/대한간학회_2021 대한간학회 비알코올 지방간질환 진료 가이드라인.pdf")
documents = loader.load()


/var/folders/2v/b65115v14cvgvjztf82wr9x80000gn/T/ipykernel_14626/2611477212.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


ValueError: File path ./data/대한간학회_2021 대한간학회 비알코올 지방간질환 진료 가이드라인.pdf is not a valid file or url

In [ ]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(
    model="embeddinggemma:300m",
    base_url="http://host.docker.internal:11434"
)
# embedding.embed_query("대한민국")


In [ ]:
from langchain_postgres import PGEngine, PGVectorStore
import os
DB_USER = os.getenv("DB_USER", "langchain")
DB_PASSWORD = os.getenv("DB_PASSWORD", "langchain")
DB_HOST = os.getenv("DB_HOST", "postgres")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "langchain")

CONNECTION_STRING = (
    f"postgresql+psycopg://"
    f"{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = PGEngine.from_connection_string(
    url=CONNECTION_STRING,
)


In [ ]:
engine.init_vectorstore_table(
    table_name = 'healths',
    vector_size=768
)

In [ ]:
vector_store= PGVectorStore.create_sync(
    engine=engine,
    table_name = "healths",
    embedding_service=embedding
)


In [ ]:
vector_store.add_documents(documents=documents)

['7f4e1aba-c96e-4bc9-83c6-6c7308712852',
 'ee19bf5a-f6ab-45ca-b046-6cabcc867aac',
 'e298f372-92b3-4a13-8d80-24ed97be46c7',
 '6e0855dd-cd3d-44e8-a7f0-36fc2d809e6c',
 '9a381ec9-1149-4bed-9f9a-31b1af121b55',
 '92d7858a-925b-4783-b0a3-7cee9f643ea9',
 '2731f96f-9084-4794-84f7-6bd8d91bee64',
 'f06b247a-0e77-4c10-98b6-620c27f9452e',
 'ce958660-074f-4543-8dd4-6dd8745305db',
 '57ca0423-406d-4fe2-bd70-5bd6d3b1de8b',
 '70026190-fc05-4107-8534-7c0e1c66e773',
 '82702470-dd4a-4ebe-b635-bf8a174dce76',
 '34dcb32a-8d3c-4818-a1bc-44a0451e5be7',
 'e2d80535-769d-43a5-96ae-b3fdaf4e5d1c',
 '85850a10-581d-4875-9746-64ed5ece6781',
 '09b5e4c3-f0f9-40c1-b381-a8080f586d99',
 '31b0138d-dbdc-44de-ad2f-b3eaaa3fbd82',
 'c05ccf95-c4ed-40ab-9856-e27f0e890737',
 '65d5aa5e-402b-47d4-adec-ce5d8e8f455c',
 'e5a34808-05f6-4f3b-850e-0b601d74f576',
 '9bc2a889-cb6b-45a1-a41e-200d0e72080e',
 '1ce65c55-3c78-486d-aa9f-33d9121a17b2',
 '58ca789b-a872-4959-ab9c-b0c92d133263',
 '69bdacde-4f9a-467d-8e71-359417f1a967',
 '4f080a1e-3216-

In [ ]:
retriever = vector_store.as_retriever(search_type='mmr', 
                                      search_kwargs={'fetch_k' : 10, 
                                      'lambda_mult' : 0.5}
                                      )
retriever.invoke("간질환과 비만의 연관성")


[Document(id='a1bc13c0-ee72-4d00-9715-2a89597b61d2', metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-04-28T11:51:30+09:00', 'moddate': '2021-05-14T10:10:30+09:00', 'trapped': '/False', 'source': './data/대한간학회_2021 대한간학회 비알코올 지방간질환 진료 가이드라인.pdf', 'total_pages': 152, 'page': 58, 'page_label': '59'}, page_content='참고문헌\n59\n2021 대한간학회 \ndisease: focus on high-risk groups. Dig Liver Dis 2015;47:997-1006.\n58. Kotronen A, Westerbacka J, Bergholm R, Pietiläinen KH, Yki-Järvinen H. Liver fat in \nthe metabolic syndrome. J Clin Endocrinol Metab 2007;92:3490-3497.\n59. Assy N, Kaita K, Mymin D, Levy C, Rosser B, Minuk G. Fatty infiltration of liver in \nhyperlipidemic patients. Dig Dis Sci 2000;45:1929-1934.\n60. Wu KT, Kuo PL, Su SB, Chen YY, Yeh ML, Huang CI, et al. Nonalcoholic fatty liver \ndisease severity is associated with the ratios of total cholesterol and triglycerides to \nhigh-density lipoprotein cholesterol. J Clin 

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()
import os


In [ ]:
api_key = os.getenv("OLLAMA_API_KEY")   

headers = {
   "Authorization": f"Bearer {api_key}"
}

llm = ChatOllama(
   base_url="https://ollama.com", # 원격 서버 주소
   model="gemma4:31b-cloud",
   client_kwargs={"headers": headers},
   temperature=0.2,
   reasoning=True
)




llm.invoke("hi")


AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user said "hi".\nThe user is initiating a conversation.\nRespond politely and offer assistance.\n\n    *   "Hello! How can I help you today?"\n    *   "Hi there! What\'s on your mind?"\n    *   "Hello! Is there anything I can assist you with?"'}, response_metadata={'model': 'gemma4:31b-cloud', 'created_at': '2026-07-23T06:19:30.627623756Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1050121819, 'load_duration': None, 'prompt_eval_count': 17, 'prompt_eval_duration': None, 'eval_count': 80, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b-cloud', 'model_provider': 'ollama'}, id='lc_run--019f8da1-0302-7241-b1bb-1b265f1ef7d4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 80, 'total_tokens': 97})

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain


In [ ]:
prompt = ChatPromptTemplate.from_template(
    """
    당신은 응급의학과와 가정의학과에서 근무하는 전문의입니다.
    환자들의 의학적 지식이 전무하다는 점을 염두하여 의학적 지식에서 사용하는 전문용어를 사용하되 쉽게 설명하고 나서 설명하는 편이다.
    최대한 환자들의 집중력을 고려하여 짧고 명확하게 설명하고 사자성어와 같이 4글자로 줄여서 외울 수 있도록 만들어 주기도 한다.
    환자들의 이해가 깊어질 수 있도록 관련한 지식을 정확하고 자세하고 매우 쉽고 짧게 설명한다는 점에서 최고의 의사이다.

    규칙 : 
    1. 제공된 정보에서 이야기 할 것 
    2. 제공된 정보의 출처를 알려줄 것
    {context} 
    
    질문 : {input}

    """
)



In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)


In [ ]:
result = rag_chain.invoke({'input' : "간질환과 비만의 연관성"})

In [ ]:
print(result['answer'])

안녕하세요. 응급의학과 및 가정의학과 전문의입니다. 

질문하신 **'비만과 간질환의 연관성'**에 대해 핵심만 짧고 명확하게 설명해 드리겠습니다.

### 📌 한 줄 요약: "살이 찌면 간에 기름이 끼고, 이것이 암까지 갈 수 있습니다."

**1. 비만은 지방간의 핵심 원인입니다**
*   **BMI(체질량 지수, 키와 몸무게로 계산한 비만도)**가 높을수록 **비알코올 지방간질환**(술을 마시지 않았는데 간에 지방이 쌓이는 병)이 생길 확률이 비례해서 높아집니다.
*   특히 **대사증후군**(복부비만, 고혈압, 고혈당, 고중성지방 등)이 있는 분들은 2명 중 1명(50%)이 지방간을 가지고 있을 정도로 연관성이 매우 깊습니다.

**2. 단순한 '기름'에서 '염증'과 '딱딱함'으로 진행됩니다**
*   단순히 지방만 낀 상태보다 **비알코올 지방간염**(지방간에 염증이 더해진 상태)이 되면 **간섬유화**(간이 흉터처럼 딱딱하게 굳는 현상)가 더 빠르게 진행됩니다.
*   이렇게 간이 굳으면 **간경변증**(간이 심하게 딱딱해진 상태)이나 **간세포암종**(간암)으로 이어질 수 있으며, 심지어 간이 딱딱해지지 않은 상태에서도 암이 발생할 수 있습니다.

**3. 위험한 결과**
*   비만으로 인한 간질환이 심해지면 간질환 자체로 사망할 뿐만 아니라, 심혈관질환이나 다른 악성종양으로 인해 사망할 위험이 커집니다.

---

### 💡 이것만은 꼭 기억하세요!
**[ 비 만 지 염 ]**
**(肥 滿 脂 炎)**
**"비만하면 $\rightarrow$ 지방이 끼고 $\rightarrow$ 염증이 생긴다"**
(비만 $\rightarrow$ 지방간 $\rightarrow$ 지방간염 순으로 악화되니 체중 관리가 필수라는 뜻입니다.)

---
**※ 출처:** 2021 대한간학회 비알코올 지방간질환 진료 가이드라인


In [ ]:
loader = PyPDFLoader("./data/대한간학회_2025년 대사이상지방간질환 진료가이드라인_최종.pdf")
documents = loader.load()
vector_store.add_documents(documents=documents)


['7b5a7167-f911-4064-9e27-00fec3762483',
 '526cb9a8-ee0d-4e3d-b672-b9817f1a37fc',
 '6b94d229-86b8-4626-8deb-5b0c9299dd66',
 '2fa1790c-19a4-43a0-83f5-f29624ef8563',
 'a94a7341-07d2-4506-9089-c767e2f8541e',
 '20842dd0-83d9-4df7-b40c-335f151673b3',
 '35fdca51-89b8-44d4-88be-0963c13ba368',
 '37390a1b-a49b-4314-81e7-c15bd6c91d4d',
 'e8b8365f-a044-4276-8b25-bc0d0123553c',
 'fca22fcb-dece-4cb2-b493-761775eb10ff',
 '85a77c1a-2854-4256-834b-6c28f4181e29',
 '95cb9e13-753f-48a3-8036-95e09ebd71ca',
 '560ae7cb-5040-49c4-9a86-6ebf02c87f4a',
 '5fbfe294-ec22-4659-9683-7249a42b9462',
 'a6ba95e4-1489-4a40-a89b-03034db07c28',
 '68490014-bdda-4dba-8995-a91e695666f1',
 '5c07d5ef-6a67-4171-98a9-e32d9f04a846',
 'ea62e599-8ec2-409b-b17b-7563de182bd7',
 '7442f432-460e-4a69-a8dc-8e220e81f835',
 '16341102-7c60-4798-bbd0-51bd02d28f7c',
 'c346395d-a8da-4639-8e9c-5b91a033cb47',
 'be11c659-bd5f-49ed-b3dd-8c2e10013d77',
 'ce7d9551-3c2c-483b-a6de-2e040def6e66',
 'df86d822-f50e-463f-8885-f57ed2bab0a7',
 'abf7081b-3fa7-

In [ ]:
result = rag_chain.invoke({'input': '2025년 대한가학회 출판일은?'})

In [ ]:
import requests
import os
from urllib.request import urlretrieve
from bs4 import BeautifulSoup
from tqdm import tqdm

_url = "https://finance.naver.com/research/company_list.naver?page={}"

In [ ]:
os.makedirs("./pdf", exist_ok=True)
pages = 10

for i in range(1, pages + 1):
    url = _url.format(i)
    bs = BeautifulSoup(requests.get(url).text)
    for x in tqdm(bs.find_all('td', class_="file")):
        try:
            # print(x.find('a')['href'])
            file = x.find('a')['href']
            urlretrieve(file, f"./pdf/{file.split("/")[-1]}")
        except:
            pass

100%|██████████| 30/30 [00:05<00:00,  5.62it/s]


In [ ]:
!du -h

238M	./pdf
4.0K	./.Trash-0/info
0	./.Trash-0/files/ollama.ipynb
0	./.Trash-0/files
4.0K	./.Trash-0
892K	./.ipynb_checkpoints
25M	./data
264M	.


In [ ]:
len(os.listdir("./pdf"))

260

In [ ]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader
)

loader = DirectoryLoader(
    path="./pdf",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

documents_pdf = loader.load()

  0%|          | 0/260 [00:00<?, ?it/s]

100%|██████████| 260/260 [01:49<00:00,  2.37it/s]


In [ ]:
len(documents_pdf)

1876

In [ ]:
# engine.init_vectorstore_table(
#     table_name = 'stocks',
#     vector_size=768
# )
vector_store_stock = PGVectorStore.create_sync(
    engine=engine,
    table_name = "stocks",
    embedding_service=embedding
)



In [ ]:
1876 // 50

37

In [ ]:
from langchain_core.utils.strings import sanitize_for_postgres
for document in documents_pdf:
    document.page_content = sanitize_for_postgres(document.page_content)

In [ ]:
# for i in range(1876 // 50 + 1):
#     vector_store_stock.add_documents(documents=documents_pdf[50 * i : 50 * (i + 1)])
#     print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37


In [ ]:
import requests
from langchain.tools import tool

@tool
def get_weather(city : str) -> str:
    """
    이 함수는 현재 도시의 날씨 정보를 리턴하는 함수입니다.
    input : city (도시 이름은 영어로) 예) seoul
    output : 해당 도시의 날씨 정보 문자열
    """
    return requests.get(f"https://wttr.in/{city}?format=j1").text

In [ ]:
@tool
def get_today() -> str:
    """
    이 함수는 현재 날짜를 리턴합니다. 
    """
    return str(date.today())


In [ ]:
from langgraph.prebuilt import create_react_agent

In [ ]:
agent = create_react_agent(
    model=llm,
    tools=[get_weather]
)


/tmp/ipykernel_65766/3979778881.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [ ]:
result = agent.invoke(
        {
        "messages" : [
            {
                "role" : 'user',
                'content' : "지금 서울의 날씨를 알려줘"
            }
        ]
    }
)